# Turtlebot Tutorial
Welcome to the tutorial. Here you will learn to:
* Read data from the robots services
* Drive the robot around
* Look at the laser scanner data
* Find way to navigate the environment
* Make the robot follow an object

Next, just execute the code cells below in sequence, read the comments, the output, and see what happens.

Don't worry, you can modify this code as you like. To save your own version, click File->Save as... and enter a new name. After you leave, the content of this notebook will automatically revert to the original state. However, if possible please re-dock the robot after use (By executing one of the cell with `robot.dock()`. In case this does not work, don't worry we will take note and help the robot back onto it's base. In case there are any questions feel free to contact us at robohub@uwaterloo.ca
If you want to take your code home with you later, click File->Make a Copy and enter a new name now. Otherwise all changes will be temporary.

In [1]:
# load ROS library Needed to start
import rclpy
if not rclpy.ok():
    rclpy.init()

# Load Robot ROS wrapper

In [2]:
# creating a function
from robot_wrapper import Robot
# Instantiate Wrapper. The subscribers and publishes the relevant topics and connects the actions.
robot = Robot()

# Example: Read data from topics

In [3]:
# Check Battery State. Note: 10% get you TODO minutes of runtime.
robot.battery_state_future = rclpy.Future()
robot.spin_until_future_completed(robot.battery_state_future);
print("Battery {0:2.3}%, charging with {1:2.2}A".format(
    robot.last_battery_state_msg.percentage*100,
    robot.last_battery_state_msg.current
))

Battery 61.0%, charging with -0.82A


In [4]:
# Wait for initial data on the subscribed topics
robot.spin_until_future_completed(robot.scan_future); # Lidar
robot.spin_until_future_completed(robot.odom_future); # Odometry
robot.spin_until_future_completed(robot.imu_future); # IMU
#robot.spin_until_future_completed(robot.image_future); # Camera image
#robot.spin_until_future_completed(robot.camera_info_future); # Camera calibration data
robot.spin_until_future_completed(robot.battery_state_future); # Battery status
# This also confirms all services are running

# Example: Drive the robot via teleoperation

In [ ]:
# Use preprogrammed behavior: Drive off the dock in a controlled way
robot.undock() # undocks

In [5]:
robot.generate_buttons()

Widget Javascript not detected.  It may not be installed or enabled properly. Reconnecting the current kernel may help.


In [6]:
robot.dock() #docks

irobot_create_msgs.action.Dock_Result(is_docked=False)

# Example: Drive a square shape

In [ ]:
robot.undock()

In [7]:
# send_cmd_vel (forward_velocity, rotation_velocity, duration)
robot.set_cmd_vel(0.1, 0.0, 2.) # 0.1
robot.set_cmd_vel(0.0, 0.5, 3.14)
robot.set_cmd_vel(0.1, 0.0, 2.) # 0.1
robot.set_cmd_vel(0.0, 0.5, 3.14)
robot.set_cmd_vel(0.1, 0.0, 2.) # 0.1
robot.set_cmd_vel(0.0, 0.5, 3.14)
robot.set_cmd_vel(0.1, 0.0, 2.) # 0.1
robot.set_cmd_vel(0.0, 0.5, 3.14)

In [ ]:
robot.dock()

# Example: Drive robot and aquire odometry
Tape measure, tape. Optional: Laser pointer.

* Tape two parallel lines `0.2m` apart on the floor. These are start and finish lines `0.2m`.
* Place the robot at the start line
* Read data from the Odometry topic
* Command the robot to drive `0.2m` straight
* Read data from the Odometry topic.
Did the robot travel `0.2m`? Where could the error come from?

Modify the experiment to investigate the behavior for rotation. Instead of the start and finish line attach a laser pointer to the base and aim it at a reference point. Command the robot to turn one full revolution.

Reading:
- Odometry sensor: TODO: find link
- Quaternions: https://en.wikipedia.org/wiki/Quaternion#Quaternions_and_the_space_geometry


In [10]:
# How to get the robot to move (forward velocity, rotational velocity and duration in s)
robot.set_cmd_vel(1,0.0,1)

In [ ]:
# Use preprogrammed behavior: Drive off the dock in a controlled way
robot.undock()

In [11]:
# Display raw data: Odometry based position
robot.odom_future = rclpy.Future()
pose1 = robot.spin_until_future_completed(robot.odom_future).pose.pose
print(pose1.position)
print(pose1.orientation)

geometry_msgs.msg.Point(x=-0.42830604314804077, y=0.4709111452102661, z=-0.002575157443061471)
geometry_msgs.msg.Quaternion(x=0.00047404365614056587, y=0.004859027918428183, z=-0.6756572723388672, w=0.7371997237205505)


In [12]:
# Drive with desired velocity command:
velocity_x = 0.1 # in m/s
velocity_phi = 0. # for rotation test use 0.5 rad/s
duration = 2. # in s
robot.set_cmd_vel(velocity_x, velocity_phi, duration)

In [13]:
# Display raw data: Odometry based position
robot.odom_future = rclpy.Future()
pose2 = robot.spin_until_future_completed(robot.odom_future).pose.pose
print(pose2.position)
print(pose2.orientation)

geometry_msgs.msg.Point(x=-0.4124072790145874, y=0.27795442938804626, z=-0.003785355482250452)
geometry_msgs.msg.Quaternion(x=-0.001072192331776023, y=0.003843804355710745, z=-0.6756959557533264, w=0.7371696829795837)


In [14]:
# translation
print("Desired distance: {}".format(velocity_x*duration))
import numpy
print("Measured distance: {}".format(
    numpy.sqrt( (pose1.position.x - pose2.position.x)**2 + (pose1.position.y - pose2.position.y)**2 ) ))

# rotation
import eigenpy # makes use of the Eigen C++ library ()
def make_quaternion(q):
    return eigenpy.Quaternion(q.w,q.x,q.y,q.z)
print("Desired rotation: {}".format(
    velocity_phi * duration))
print("Measured rotation: {}".format(
    make_quaternion(pose1.orientation).angularDistance(make_quaternion(pose2.orientation))))


Desired distance: 0.2
Measured distance: 0.1936106011608561
Desired rotation: 0.0
Measured rotation: 0.0037007697882682922


In [ ]:
robot.dock()

# Example: Find the Tag, Detect April tags with OpenCV
For this example, you need a printed April tag label from the series 36h11. Alternatively change the code. To generate numerically useful results and for verification, a ruler or tape measure is necessary as well.

In [ ]:
robot.undock()

In [ ]:
# Put marker in front of camera
# Take image and convert to in opencv format
robot.image_future = rclpy.Future()
robot.spin_until_future_completed(robot.image_future)

import cv2 # important to prevent triggering a bug when loading cv_bridge
from cv_bridge import CvBridge, CvBridgeError
bridge = CvBridge()
try:
    cv_image = bridge.imgmsg_to_cv2(robot.last_image_msg, "passthrough")
except CvBridgeError as e:
    rospy.logerr("CvBridge Error: {0}".format(e))

In [ ]:
# Display image
import cv2
rgb_image = cv2.cvtColor(cv_image, cv2.COLOR_BGR2RGB)
from matplotlib import pyplot as plt
plt.imshow(rgb_image)
plt.show()

In [ ]:
# Detect aruco markers
import cv2
import numpy
robot.spin_until_future_completed(robot.camera_info_future)
adict = cv2.aruco.Dictionary_get(cv2.aruco.DICT_APRILTAG_36h11)
res = cv2.aruco.detectMarkers(
    cv_image,
    dictionary=adict,
    cameraMatrix=robot.last_camera_info_msg.k.reshape((3,3)),
    distCoeff=numpy.array(robot.last_camera_info_msg.d)
)
print("Found ids: {}".format(res[1]))

In [ ]:
# Plot detected markers in picture
frame_markers = cv2.aruco.drawDetectedMarkers(cv_image.copy(), res[0], res[1])
from matplotlib import pyplot as plt
plt.imshow(frame_markers)
plt.show()

In [ ]:
# Compute marker position in camera frame
rvec, tvec, markerPoints = cv2.aruco.estimatePoseSingleMarkers(
    res[0],
    0.08,
    robot.last_camera_info_msg.k.reshape((3,3)),
    numpy.array(robot.last_camera_info_msg.d)
    )
print(tvec)

# Example: Follow the tag

In [ ]:
robot.spin_until_future_completed(robot.image_future) 
robot.spin_until_future_completed(robot.camera_info_future)
# Take image and convert to in opencv format
import cv2 # important to prevent triggering a bug when loading cv_bridge
import numpy
from cv_bridge import CvBridge, CvBridgeError
bridge = CvBridge()

def get_marker_position():
    # get new image
    robot.image_future = rclpy.Future()
    robot.spin_until_future_completed(robot.image_future)
    cv_image = bridge.imgmsg_to_cv2(robot.last_image_msg, "passthrough")
    
    # Detect aruco markers
    import cv2
    adict = cv2.aruco.Dictionary_get(cv2.aruco.DICT_APRILTAG_36h11)
    res = cv2.aruco.detectMarkers(
        cv_image,
        dictionary=adict,
        cameraMatrix=robot.last_camera_info_msg.k.reshape((3,3)),
        distCoeff=numpy.array(robot.last_camera_info_msg.d)
    )
    #print("Found ids: {}".format(res[1]))
    if len(res[0])==0:
        raise Exception("No marker visible")
    # Compute marker position in camera frame
    rvec, tvec, markerPoints = cv2.aruco.estimatePoseSingleMarkers(
        res[0],
        0.08,
        robot.last_camera_info_msg.k.reshape((3,3)),
        numpy.array(robot.last_camera_info_msg.d)
        )
    return tvec

In [ ]:
Kp_translation = 1. #3
Kp_heading = 0.2 #5
distance_ref = 0.6
duration = 0.2
while True:
    try:
        tvec = get_marker_position().reshape(3)
        print("Marker positon",tvec)

        # compute forward velocity given distance to marker
        e_translation =  tvec[2] - distance_ref
        if abs(e_translation) < 0.02:
            forward_velocity = 0
        else:
            forward_velocity = min(Kp_translation * e_translation,0.1)
        print("Forward velocity",forward_velocity)

        # compute rotation velocity given marker offset
        e_heading = - tvec[0]
        if abs(e_heading) < 0.05:
            rotation_velocity = 0
        else:
            rotation_velocity = Kp_heading * e_heading
        print("Rotation velocity", rotation_velocity)

        duration = 0.25 # in s
        robot.set_cmd_vel(forward_velocity, rotation_velocity, duration)
    except Exception as ex:
        print("Error",ex)
        import time
        time.sleep(1.)

# Example: Drive robot and aquire odometry and lidar data

In [ ]:
robot.undock()

In [ ]:
# Display data: Lidar scan data
print("timestamp: {}\nangle_min: {}\nangle_max: {}\nangle_increment: {}\nranges (first 3): {}".format(
      robot.last_scan_msg.header.stamp,
      robot.last_scan_msg.angle_min,
      robot.last_scan_msg.angle_max,
      robot.last_scan_msg.angle_increment,
      robot.last_scan_msg.ranges[:3]))

In [ ]:
from matplotlib import pyplot as plt
# Plot robot position from odometry
# Use blue cross marker to designate the robot position
pose = robot.last_odom_msg.pose.pose
plt.plot([pose.position.x],[pose.position.y],'bx') # TODO: show robot front

# Plot lidar points from last message
def plot_lidar_scan_points(msg,pose):
    # Convert received lidar points into cartesian coordinates considering the current position from odometry
    import numpy
    points = [(numpy.cos(angle)*radius,numpy.sin(angle)*radius) for angle, radius \
              in zip(numpy.linspace(msg.angle_min,msg.angle_max,len(msg.ranges)),msg.ranges)]

    T = robot.reduce_transform_to_2D(robot.convert_odom_to_transform(pose))
    
    # remove points which cannot be measured
    filtered_points = filter(lambda x: numpy.isfinite(x).all(),points)
    
    # display points in world frame
    transformed_points = [numpy.matmul(T,
        numpy.vstack([ numpy.atleast_2d(x).T,numpy.ones((1,1)) ]) ) for x in 
                          filtered_points]
    plt.plot(
        [x[0] for x in transformed_points],
        [x[1] for x in transformed_points],'r.')

plot_lidar_scan_points(robot.last_scan_msg,pose)

In [ ]:
# Drive robot and plot repeatedly into same plot
# TODO: this should illustrate the odometry drift and motivate slam
# TODO: check that time delays are not the source of observable error

for idx in range(4):
    # ensure fresh data
    robot.scan_future = rclpy.Future()
    robot.odom_future = rclpy.Future()
    robot.spin_until_future_completed(robot.scan_future);
    robot.spin_until_future_completed(robot.odom_future);
    
    pose = robot.last_odom_msg.pose.pose
    plt.plot([pose.position.x],[pose.position.y],'bx') # TODO: show robot front
    plot_lidar_scan_points(robot.last_scan_msg,pose)
    print("Time delta {}s".format(
        (rclpy.time.Time.from_msg(robot.last_odom_msg.header.stamp)
            -
        rclpy.time.Time.from_msg(robot.last_scan_msg.header.stamp)).nanoseconds * 1e-9  ) ) 
    
    # Drive a bit
    robot.set_cmd_vel(0.2,0.,4.)

As likely visible in the plot above, the lidar reflections are not in the same spots. This is because of odometry error. It motivates the use of data fusion to create a better map.

In [ ]:
# Turn approximately 180deg
robot.set_cmd_vel(0.,1.,2.8)
# Drive back to base
robot.set_cmd_vel(0.2,0.,12)

In [ ]:
# Drive onto the dock in a controlled way
robot.dock()

# Example: Compute travelled distance from Lidar data
For this experiment, the environment has to present enough features to the lidar.

In [ ]:
# Undock
robot.undock()

In [ ]:
# Aquire data at point 1
robot.scan_future = rclpy.Future()
robot.odom_future = rclpy.Future()
robot.spin_until_future_completed(robot.scan_future);
robot.spin_until_future_completed(robot.odom_future);
scan1 = robot.last_scan_msg
pose1 = robot.last_odom_msg

In [ ]:
# Drive a bit
robot.set_cmd_vel(0.1,0.,1.)

In [ ]:
# Aquire data at point 2
robot.scan_future = rclpy.Future()
robot.odom_future = rclpy.Future()
robot.spin_until_future_completed(robot.scan_future);
robot.spin_until_future_completed(robot.odom_future);
scan2 = robot.last_scan_msg
pose2 = robot.last_odom_msg

In [ ]:
def msg2transform(msg):
    mat = numpy.zeros((3,3))
    mat[0:2,0:2] = numpy.eye(2) # TODO: rotation from quaternion
    mat[0,2] = msg.pose.pose.position.x
    mat[1,2] = msg.pose.pose.position.y
    mat[2,2] = 1.
    return mat
pose1_mat = msg2transform(pose1)
pose2_mat = msg2transform(pose2)
print("pose1\n",pose1_mat)
print("pose2\n",pose2_mat)

In [ ]:
# Compute pose difference of two lidar distances

def pose_offset_from_images(scan1,scan2):
    import numpy
    msg = scan1
    points1 = [(numpy.cos(angle)*radius,numpy.sin(angle)*radius) for angle, radius \
              in zip(numpy.linspace(msg.angle_min,msg.angle_max,len(msg.ranges)),msg.ranges)]
    msg = scan2
    points2 = [(numpy.cos(angle)*radius,numpy.sin(angle)*radius) for angle, radius \
              in zip(numpy.linspace(msg.angle_min,msg.angle_max,len(msg.ranges)),msg.ranges)]
    import cv2
    mat = numpy.zeros((3,3))
    mat[0:2,0:3] = cv2.estimateAffine2D(numpy.array(points1),numpy.array(points2))[0]
    mat[2,2] = 1.
    return mat

lidar_transform = pose_offset_from_images(scan1,scan2)
print("Lidar Transform:\n{}".format(lidar_transform))
# TODO: put transform into odometry frame
odometry_transform = numpy.dot(numpy.linalg.inv(pose1_mat),pose2_mat)
print("Odometry Transform:\n{}".format(odometry_transform))

# TODO: distance travelled is half what odometry reports, unclear why
print("Distance travelled:\nLidar: {}\nOdometry: {}".format(
    numpy.linalg.norm(lidar_transform[:,2]),
    numpy.linalg.norm(pose1_mat[:,2] - pose2_mat[:,2])
))

In [ ]:
robot.dock()

# Example: Wall following
Learn how to interpret lidar data and control the robot with it.

In [ ]:
# e.g. with cv2.fitLine(points, distType, param, reps, aeps[, line]) → line
# 1. aquire
# 2. fit line in static context
# 3. compute command from line data
# 4. run in loop

# Example: Data fusion - Luenberger Observer with Odometry and Lidar

In [ ]:
# TODO: WIP
# TODO: convert to homogeneous transformations to simplify math
# TODO: 2D exponential twist -> homogeneous transformation
# Filter gain (magic)
gain = 0.1
# Desired velocity for testing
velocity_x = 0.1
velocity_phi = 0.
# 
experiment_duration = 1.

# Filter state (output)
# Initialize with estimated pose with current pose of odometry (to be able to compare both)
estimated_pose = robot.last_odom_msg.pose.pose

# drive continously while aquiring data
sample_times = [time.time()]
lidar_scans = [robot.last_scan_msg]
lidar_poses = [estimated_pose]

while time.time() > sample_times[0] + experiment_duration:
    t = rclpy.time.Time(time.time())
    dt = t - sample_times[-1]
    sample_times.append(t)
    # Prediction step, use desired velocity to estimate the displacement
    estimated_pose.position = \
        estimated_pose.position + [sin(phi) * velocity_x, cos(phi) * velocity_x, 0.] * dt
    # TODO: apply quaternion magic
    estimated_pose.orientation = \
        estimated_pose.orientation + [velocity_phi] * dt
    # Correction step, compute a better pose from a reliable source
    lidar_offset = pose_offset_from_images(robot.last_scan_msg,lidar_scans[-1])
    lidar_pose = lidar_offset + lidar_poses[-1]
    lidar_scans.append(robot.last_scan_msg)
    lidar_poses.append(lidar_pose)
    # Apply correct to filter state
    estimated_pose.position = gain * ( lidar_pose.position - estimated_pose.position)
    # TODO: add quaternion magic
    #estimated_pose.orientation = gain * ( lidar_pose.orientation - estimated_pose.orientation)

In [ ]:
# Drive robot and plot repeatedly into same plot
# TODO: Drive robot with same code as above, but use filter to estimate position.
# TODO: Scan should now match


# Example: Mini-SLAM
TODO: is this realistic?

In [ ]:
# drive and aquire lidar while accumulating map data using pose_offset_from_images

# Example: Dynamic obstacle avoidance with potential fields

# Example: Spline based path planning on how to approach april tag

# Example: Visual servoing with template matching

In [ ]:
# TODO: detect object with cloud based service like
# https://cloud.google.com/vision/docs/object-localizer#vision_localize_objects-python
# use RGBD-data to discover depth and demonstrate similar application.